In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad

# adata = ad.read_h5ad("./data/larry/postprocessed.h5ad")
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import joblib
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import *

# Load the object back
emb = joblib.load("./data/larry/larry_embedder.pkl")

In [ ]:
neut_mask = adata.obs["state_info"] == "Neutrophil"
X_neut = emb.X_raw[neut_mask.values]
V_neut = emb.V_raw[neut_mask.values]

# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb_neut = VectorFieldEmbedder(
    X_neut, V_neut,
    dist_method="phase",
    dof=50,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb_neut.initialize_embedding(seed=123)

plot_vector_field_grid(emb_neut.X_emb,
    tps_vf=emb_neut.tps_vf, grid_size=25)

In [ ]:
emb_neut.fit_gene_level_splines(dof_gene=100, dof_vf_gene=100)

# Compute local Jacobian J = ∂ψ/∂y (maps embedding → gene expression)
jacobians = emb_neut.tps_gene.compute_jacobians(emb_neut.X_emb)  # (n_cells, n_genes, d)

In [ ]:
fit_stats = emb_neut.tps_gene.evaluate_fit(emb_neut.X_emb, emb_neut.X_raw)

p_adj = fit_stats["p_adj"]
neglogp = -fit_stats["log_p_adj"]
mean_expr = np.mean(emb_neut.X_raw, axis=0)

thr = -np.log(0.05)

plt.figure(figsize=(6, 5))
sns.set_style("white")
sns.set_context("talk")  # larger base font scaling

sns.scatterplot(
    x=np.log1p(mean_expr),
    y=neglogp,
    s=40, alpha=0.7,
    edgecolor="none", color="teal"
)

plt.axhline(thr, color="gray", ls="--", lw=1)
plt.text(
    x=np.min(np.log1p(mean_expr)) + 0.1,
    y=thr + 0.8,
    s="FDR = 0.05",
    color="gray",
    fontsize=13,
    va="bottom"
)

plt.xlabel("log(mean expression + 1)", fontsize=14)
plt.ylabel("−log(FDR-adjusted p)", fontsize=14)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.ylim(0, 50)
plt.xlim(np.min(np.log1p(mean_expr)) - 0.2, np.max(np.log1p(mean_expr)) + 0.2)

sns.despine(trim=True)
plt.tight_layout()
plt.show()


In [ ]:
cell_idx = 30
V = V_neut[cell_idx]
J = jacobians[cell_idx]

# --- Project observed velocity into tangent subspace ------------------
# Handle potential rank deficiency gracefully
JTJ = J.T @ J
if np.linalg.cond(JTJ) > 1e8:
    print("⚠️ Ill-conditioned Jacobian, using pseudoinverse")
    JTJ_inv = np.linalg.pinv(JTJ)
else:
    JTJ_inv = np.linalg.inv(JTJ)

P_tangent = J @ JTJ_inv @ J.T   # projection operator to tangent subspace
V_tangent = P_tangent @ V       # projected velocity

# --- Sweep directions in tangent space --------------------------------
n_directions = 180
angles = np.linspace(0, 2*np.pi, n_directions, endpoint=False)
mse_values = np.zeros((n_directions, J.shape[0]))  # (angles × genes)

for k, theta in enumerate(angles):
    d = np.array([np.cos(theta), np.sin(theta)])
    basis_vec = J @ d   # (n_genes,)
    denom = np.dot(basis_vec, basis_vec)
    if denom < 1e-12:
        mse_values[k, :] = np.nan
        continue

    # --- Fit coefficient using raw velocity V --------------------------
    a = np.dot(V_tangent, basis_vec) / denom
    a = np.maximum(a, 0)
    V_pred = a * basis_vec

    # --- Compute per-gene squared error (still in gene space) ----------
    mse_values[k, :] = (V_tangent - V_pred) ** 2
    gene_var = V_tangent ** 2
    mse_values[k, :] /= (gene_var + 1e-12)
    
# --- Compute per-gene MSE normalized by velocity magnitude -------------
threshold = 0.5  # tune this threshold

fit_mask = mse_values < threshold  # (n_directions, n_genes)
fit_counts = np.sum(fit_mask, axis=1)         # number of good genes per direction

# --- Circular histogram ------------------------------------------------
fig = plt.figure(figsize=(6,6), facecolor="white")
ax = fig.add_subplot(111, polar=True)

bar_width = 2 * np.pi / n_directions
ax.bar(angles, fit_counts, width=bar_width, bottom=0,
       color="teal", alpha=0.6, edgecolor="none")

ax.set_theta_zero_location("E")
ax.set_theta_direction(-1)
ax.set_title(f"Directional coherence (# good genes per angle, cell {cell_idx})", pad=20)
ax.set_rticks([])
plt.show()

In [ ]:
X = emb_neut.X_emb
highlight = np.atleast_1d(cell_idx)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c='lightgray', s=20, alpha=0.5, label='All cells')
plt.scatter(X[highlight, 0], X[highlight, 1],
            c='red', s=80, edgecolor='black', label='Selected cell(s)')

plt.title("Embedding with highlighted cell(s)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.axis('equal')
plt.show()